# SEOCHO Quickstart

Define your schema → Index your data → Query your graph.

No server needed. Everything runs locally.

In [ ]:
# Install (run once)
# !pip install seocho neo4j openai

## 1. Define Your Ontology

An ontology tells SEOCHO what entities and relationships exist in your domain.

In [ ]:
from seocho import Ontology, NodeDef, RelDef, P

ontology = Ontology(
    name="news",
    description="News article knowledge graph",
    nodes={
        "Person": NodeDef(
            description="A public figure",
            properties={"name": P(str, unique=True), "title": P(str)},
        ),
        "Company": NodeDef(
            description="A business entity",
            properties={"name": P(str, unique=True), "sector": P(str)},
        ),
        "Event": NodeDef(
            description="A named event",
            properties={"name": P(str, unique=True), "date": P(str)},
        ),
    },
    relationships={
        "WORKS_AT": RelDef(source="Person", target="Company", cardinality="MANY_TO_ONE"),
        "INVOLVED_IN": RelDef(source="Person", target="Event"),
    },
)

print(ontology)

## 2. Inspect the Ontology

See what prompts, constraints, and validation rules are derived from your schema.

In [ ]:
# What the LLM sees during extraction
ctx = ontology.to_extraction_context()
print("Entity types:")
print(ctx["entity_types"])
print()
print("Relationship types:")
print(ctx["relationship_types"])

In [ ]:
# What the LLM sees during querying
ctx = ontology.to_query_context()
print(ctx["graph_schema"])
print()
print("Query hints:")
print(ctx["query_hints"])

In [ ]:
# SHACL validation shapes (auto-derived)
import json
shacl = ontology.to_shacl()
print(json.dumps(shacl, indent=2))

In [ ]:
# Cypher constraints that will be applied to the database
for stmt in ontology.to_cypher_constraints():
    print(stmt)

## 3. Save as JSON-LD

JSON-LD is the canonical storage format. Commit this to version control.

In [ ]:
doc = ontology.to_jsonld("schema.jsonld")
print(json.dumps(doc, indent=2, ensure_ascii=False))

## 4. Connect and Index

Requires a running Neo4j/DozerDB instance and an OpenAI API key.

In [ ]:
from seocho import Seocho
from seocho.store import Neo4jGraphStore, OpenAIBackend

s = Seocho(
    ontology=ontology,
    graph_store=Neo4jGraphStore("bolt://localhost:7687", "neo4j", "password"),
    llm=OpenAIBackend(model="gpt-4o"),
)

# Apply schema constraints
print(s.ensure_constraints())

In [ ]:
# Index some articles
articles = [
    "Apple CEO Tim Cook announced new AI features at WWDC 2025 in Cupertino.",
    "Samsung's Jay Y. Lee met with NVIDIA's Jensen Huang in Seoul to discuss chip supply.",
    "The European Commission fined Meta 1.2B euros for data transfer violations.",
]

for article in articles:
    mem = s.add(article, database="news_kg")
    print(f"Indexed: {mem.metadata.get('nodes_created', 0)} nodes, {mem.metadata.get('relationships_created', 0)} rels")

## 5. Extract Without Writing (Inspection)

In [ ]:
# Preview what gets extracted before writing
result = s.extract("Elon Musk is the CEO of Tesla and SpaceX.")
print(json.dumps(result, indent=2))

In [ ]:
# Score extraction quality
scores = ontology.score_extraction(result)
print(f"Overall quality: {scores['overall']:.1%}")
for node in scores["nodes"]:
    print(f"  {node['label']}({node['id']}): {node['score']:.1%} — {node['details']}")

In [ ]:
# Validate against SHACL
errors = ontology.validate_with_shacl(result)
if errors:
    for e in errors:
        print(f"  Issue: {e}")
else:
    print("  No validation issues")

## 6. Query

In [ ]:
answer = s.ask("Who is the CEO of Apple?", database="news_kg")
print(answer)

In [ ]:
# With reasoning mode (auto-retry if first query fails)
answer = s.ask(
    "What happened in Seoul?",
    database="news_kg",
    reasoning_mode=True,
    repair_budget=2,
)
print(answer)

In [ ]:
# Raw Cypher query
records = s.query(
    "MATCH (p:Person)-[:WORKS_AT]->(c:Company) RETURN p.name AS person, c.name AS company",
    database="news_kg",
)
for r in records:
    print(f"  {r['person']} works at {r['company']}")

## 7. Index from Files

Drop `.txt`, `.csv`, `.json`, `.jsonl` files in a directory and index them all.

In [ ]:
# Index a directory (uncomment to run)
# result = s.index_directory("./my_data/", database="news_kg")
# print(f"Indexed {result['files_indexed']} files, {result['files_unchanged']} unchanged")

## 8. Denormalization (Flatten for Export)

In [ ]:
# See what's safe to flatten
plan = ontology.denormalization_plan()
for label, info in plan.items():
    for embed in info["embeds"]:
        status = "SAFE" if embed["safe"] else "BLOCKED"
        print(f"  {label} -[{embed['via']}]-> {embed['target']}: {status}")

In [ ]:
# Flatten graph data
nodes = [
    {"id": "p1", "label": "Person", "properties": {"name": "Tim Cook", "title": "CEO"}},
    {"id": "c1", "label": "Company", "properties": {"name": "Apple", "sector": "Tech"}},
]
rels = [{"source": "p1", "target": "c1", "type": "WORKS_AT", "properties": {}}]

flat = ontology.to_denormalized_view(nodes, rels)
for record in flat:
    print(f"{record['label']}({record['id']}): {record['properties']}")

## 9. Cleanup

In [ ]:
s.close()
print("Done.")